In [2]:
!pip install mordred

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 2.4 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.0 MB 2.6 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 3.1 MB/s eta 0:00:00
  Created wheel for mordred: filename=mordred-1.2.0-py3-none-any.whl size=176808 sha256=b4f431ce56fcc991bae240f1a1b3980c35b3a520d02ad40901b52224fb080641
  Stored in directory: c:\users\fahad\appdata\local\pip\cache\wheels\05\95\d1\9e913738f0e8362b3676917b953a60afd76d2b0b99ff8a71ec
Successfully built mordred
  Attempting uninstall: networkx
    Found existing installation: networkx 3.2.1
    Uninstalling networkx-3.2.1:
      Successfully uninstalled networkx-3.2.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dgllife 0.3.0 requires scikit-learn<1.0,>=0.22.2, but you have scikit-learn 1.1.3 which is incompatible.


In [3]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem import Descriptors
from tqdm import tqdm
import logging
from func_timeout import func_timeout, FunctionTimedOut
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import feyn
from mordred import Calculator, descriptors

logging.basicConfig(filename='descriptor_errors.log', level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')

In [4]:
acidic_file = "acidic_pKa_cleaned_final.csv"
smiles_col = 'SMILES'
target = 'pka_value'

df = pd.read_csv(acidic_file)
df[target] = pd.to_numeric(df[target], errors='coerce')

In [5]:
saltRemover = SaltRemover(defnFilename='./Salts.txt')
df['Mol'] = df[smiles_col].astype(str).apply(
    lambda s: saltRemover.StripMol(Chem.MolFromSmiles(s))
)

In [6]:
calc = Calculator(descriptors, ignore_3D=True)

def safe_call(func, mol, timeout=1):
    try:
        return func_timeout(timeout, func, args=(mol,))
    except (FunctionTimedOut, Exception) as e:
        logging.info(f"Error in {func.__name__}: {e}")
        return np.nan

def compute_rdkit_descriptors(mol):
    descriptor_funcs = {name: func for name, func in Descriptors.descList}
    if mol is None or Chem.MolToSmiles(mol) == '':
        return None
    return {name: safe_call(func, mol, timeout=1) for name, func in descriptor_funcs.items()}

def compute_mordred_descriptors(mols):
    try:
        return calc.pandas(mols)
    except Exception as e:
        logging.info(f"Mordred batch error: {e}")
        return pd.DataFrame()

def compute_descriptors_for_df(df):
    mols = df['Mol'].tolist()
    rdkit_list = []
    for mol in tqdm(mols, desc="Computing RDKit descriptors"):
        rdkit_desc = compute_rdkit_descriptors(mol)
        rdkit_list.append(rdkit_desc if rdkit_desc is not None else {})
    rdkit_df = pd.DataFrame(rdkit_list)
    mordred_df = compute_mordred_descriptors(mols)
    full_desc_df = pd.concat([rdkit_df, mordred_df], axis=1)
    full_desc_df = full_desc_df.loc[:, full_desc_df.std(numeric_only=True) > 0]
    full_desc_df = full_desc_df.apply(pd.to_numeric, errors='coerce')
    full_df = pd.concat([df[[target]].reset_index(drop=True), full_desc_df.reset_index(drop=True)], axis=1)
    return full_df.dropna()

In [ ]:
data = compute_descriptors_for_df(df)
train_data, test_data = train_test_split(data, test_size=0.25, random_state=42)

  2%|█▎                                                                             | 158/9191 [00:02<01:30, 99.37it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Autocorrelation.py:97: RuntimeWarning: Mean of empty slice.
  return avec - avec.mean()
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Constitutional.py:80: RuntimeWarning: invalid value encountered in double_scalars
  return S / self.mol.GetNumAtoms()


 17%|█████████████▍                                                                | 1579/9191 [00:19<02:54, 43.61it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 18%|█████████████▊                                                                | 1629/9191 [00:20<01:35, 79.10it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 22%|█████████████████▍                                                            | 2059/9191 [00:27<01:42, 69.46it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 57%|████████████████████████████████████████████▎                                 | 5228/9191 [01:02<00:41, 96.58it/s]

In [ ]:
ql = feyn.QLattice()
models = ql.auto_run(train_data, output_name=target, n_epochs=200, threads=16, max_complexity=200, criterion='bic')

best_model = models[0]
y_true = test_data[target].values
y_pred = best_model.predict(test_data)
r2 = r2_score(y_true, y_pred)

print("\n--- QLattice Performance (RDKit + Mordred descriptors) ---")
print(f"Test R²: {r2:.3f}")
best_model.plot(train_data, test_data)
best_model.plot_regression(test_data)